[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/05_ONNX_Operators_and_OpSets/03_Custom_Operators/Custom_Operators_Deep_Dive.ipynb)

# 5.3 Custom Operators — Deep Dive

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [When You Need Custom Operators](#1-when-you-need-custom-operators) | Decision framework: standard ops vs custom ops vs FunctionProto |
| 2 | [Custom Operator Schema Definition](#2-custom-operator-schema-definition) | Formal contract specification |
| 3 | [Registration: Model-Side Declaration](#3-registration-model-side) | Declaring custom ops in the ONNX graph |
| 4 | [Registration: Runtime-Side Kernels](#4-registration-runtime-side) | Kernel registration in ONNX Runtime |
| 5 | [Type Inference Rules](#5-type-inference-rules) | Formal type propagation for custom ops |
| 6 | [Shape Inference Rules](#6-shape-inference-rules) | Custom shape inference functions |
| 7 | [FunctionProto as an Alternative](#7-functionproto-alternative) | Decomposing custom ops into standard ops |
| 8 | [End-to-End Workflow](#8-end-to-end-workflow) | Complete 8-step custom op lifecycle |
| 9 | [Mixed Standard + Custom Op Models](#9-mixed-models) | Building hybrid graphs |
| 10 | [Testing Patterns](#10-testing-patterns) | Golden vectors, fuzz testing, parity checks |
| 11 | [Security Considerations](#11-security-considerations) | Risks and mitigations |
| 12 | [Key Takeaways](#12-key-takeaways) | Summary |

In [ ]:
# !pip install onnx numpy --quiet

import numpy as np
import onnx
from onnx import helper, TensorProto, checker, numpy_helper, defs
from onnx import shape_inference

print(f"ONNX version: {onnx.__version__}")
print(f"Default opset: {defs.onnx_opset_version()}")

<a id='1-when-you-need-custom-operators'></a>
## 1. When You Need Custom Operators

Custom operators bridge the gap between what the ONNX standard defines and what your
model actually computes. Before creating one, consider the alternatives:

### Decision Framework

```
  Can your computation be expressed with standard ONNX ops?
      │
      ├── YES ──▶ Use standard ops (maximum portability)
      │
      └── NO ──▶ Can it be decomposed into a subgraph of standard ops?
                    │
                    ├── YES ──▶ Use FunctionProto (portable decomposition)
                    │
                    └── NO ──▶ Is performance critical?
                                  │
                                  ├── YES ──▶ Custom op with optimized kernel
                                  │
                                  └── NO ──▶ Custom op with reference kernel
```

### When Custom Ops Are Justified

| Scenario | Why Custom Op Helps | Example |
|----------|---------------------|----------|
| **No standard equivalent** | Operation doesn't exist in ONNX | Flash Attention, RMS Norm (pre-opset 21) |
| **Performance** | Fused kernel beats decomposition | BiasGelu, FusedMatMul |
| **Numerical parity** | Match training implementation exactly | Custom loss functions |
| **Proprietary IP** | Ship black-box op with visible schema | Proprietary activation |
| **Hardware-specific** | Leverage hardware features | Custom quantization kernels |

### When NOT to Use Custom Ops

| Situation | Better Alternative |
|-----------|-------------------|
| Short subgraph of standard ops suffices | Standard ops (max portability) |
| Need broad interoperability | FunctionProto decomposition |
| ONNX Functions can express the logic | FunctionProto (portable) |
| Temporary workaround for missing op | Check latest opset first |

### The Portability-Performance Tradeoff

$$\text{Portability} \propto \frac{1}{\text{Custom Ops Count}}$$

Every custom op requires a matching runtime kernel on every target platform.
Standard ops work everywhere; custom ops only work where you ship kernels.

In [ ]:
# Example: RMSNorm can be built from standard ops (no custom op needed)
# RMSNorm(x) = x / sqrt(mean(x^2) + eps) * gamma

def build_rmsnorm_standard_ops(hidden_size, eps=1e-6):
    """Build RMSNorm using only standard ONNX operators."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", "seq", hidden_size])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

    gamma = numpy_helper.from_array(
        np.ones(hidden_size, dtype=np.float32), "gamma")
    eps_val = numpy_helper.from_array(
        np.array(eps, dtype=np.float32), "eps")
    axes_val = numpy_helper.from_array(
        np.array([-1], dtype=np.int64), "axes")

    nodes = [
        helper.make_node("Mul", ["X", "X"], ["x_sq"]),
        helper.make_node("ReduceMean", ["x_sq", "axes"], ["mean_sq"], keepdims=1),
        helper.make_node("Add", ["mean_sq", "eps"], ["mean_sq_eps"]),
        helper.make_node("Sqrt", ["mean_sq_eps"], ["rms"]),
        helper.make_node("Div", ["X", "rms"], ["normed"]),
        helper.make_node("Mul", ["normed", "gamma"], ["Y"]),
    ]

    graph = helper.make_graph(
        nodes, "rmsnorm", [X], [Y],
        initializer=[gamma, eps_val, axes_val])
    model = helper.make_model(
        graph, opset_imports=[helper.make_opsetid("", 17)])
    return shape_inference.infer_shapes(model)

rmsnorm_model = build_rmsnorm_standard_ops(64)
checker.check_model(rmsnorm_model)

print(f"RMSNorm with standard ops:")
print(f"  Nodes: {len(rmsnorm_model.graph.node)}")
print(f"  Ops:   {[n.op_type for n in rmsnorm_model.graph.node]}")

try:
    from onnx.reference import ReferenceEvaluator
    ev = ReferenceEvaluator(rmsnorm_model)
    x = np.random.randn(2, 4, 64).astype(np.float32)
    y = ev.run(None, {"X": x})[0]
    rms_manual = x / np.sqrt(np.mean(x**2, axis=-1, keepdims=True) + 1e-6)
    print(f"  Input shape:  {x.shape}")
    print(f"  Output shape: {y.shape}")
    print(f"  Matches manual: {np.allclose(y, rms_manual, atol=1e-5)}")
except ImportError:
    print("  ReferenceEvaluator not available")

<a id='2-custom-operator-schema-definition'></a>
## 2. Custom Operator Schema Definition

A custom operator schema is a **formal contract** that specifies:

### Schema Components

```
┌─────────────────────────────────────────────────────────────────────┐
│                   CUSTOM OPERATOR SCHEMA                           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  1. IDENTITY                                                        │
│     ├── domain:    "com.myorg.ops"     (reverse-DNS namespace)      │
│     ├── op_type:   "RmsNorm"           (ASCII identifier)           │
│     └── version:   1                   (domain-local version)       │
│                                                                     │
│  2. INPUTS                                                          │
│     ├── X:     T    (required, single)  — input tensor              │
│     └── gamma: T    (required, single)  — scale parameter           │
│                                                                     │
│  3. OUTPUTS                                                         │
│     └── Y:     T    (required, single)  — normalized output         │
│                                                                     │
│  4. ATTRIBUTES                                                      │
│     └── epsilon: float (default=1e-6)  — numerical stability        │
│                                                                     │
│  5. TYPE CONSTRAINTS                                                │
│     └── T ∈ {float16, float32, float64}                             │
│                                                                     │
│  6. SHAPE INFERENCE                                                 │
│     └── output.shape = input.shape                                  │
│                                                                     │
│  7. DOCUMENTATION                                                   │
│     └── "Computes x / sqrt(mean(x²) + eps) * gamma"                │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Formal Notation for Schema Definition

A schema $S$ for operator $O$ is a tuple:

$$S(O) = (\mathcal{I}, \mathcal{O}, \mathcal{A}, \mathcal{T}, \mathcal{F}_{\text{shape}}, \mathcal{D})$$

where:
- $\mathcal{I} = \{(\text{name}_i, T_i, \text{arity}_i)\}$ — input formal parameters
- $\mathcal{O} = \{(\text{name}_j, T_j, \text{arity}_j)\}$ — output formal parameters
- $\mathcal{A} = \{(\text{name}_k, \text{type}_k, \text{default}_k, \text{required}_k)\}$ — attributes
- $\mathcal{T} = \{(T_v, \text{allowed\_types}_v)\}$ — type variable constraints
- $\mathcal{F}_{\text{shape}}$ — shape inference function
- $\mathcal{D}$ — documentation string

### Input/Output Arity

| Arity | Meaning | ONNX Enum |
|-------|---------|----------|
| **Single** | Exactly one tensor | `FormalParameterOption.Single` |
| **Optional** | Zero or one tensor | `FormalParameterOption.Optional` |
| **Variadic** | One or more tensors | `FormalParameterOption.Variadic` |

In [ ]:
# Define a custom operator schema formally
CUSTOM_DOMAIN = "com.tutorial.ops"
CUSTOM_OPSET = 1

# Schema definition for our custom RmsNorm operator
custom_schema = {
    "domain": CUSTOM_DOMAIN,
    "op_type": "RmsNorm",
    "version": CUSTOM_OPSET,
    "inputs": [
        {"name": "X", "type_var": "T", "arity": "Single",
         "description": "Input tensor of shape [*, hidden_size]"},
        {"name": "gamma", "type_var": "T", "arity": "Single",
         "description": "Scale parameter of shape [hidden_size]"},
    ],
    "outputs": [
        {"name": "Y", "type_var": "T", "arity": "Single",
         "description": "Normalized output, same shape as X"},
    ],
    "attributes": [
        {"name": "epsilon", "type": "float", "default": 1e-6,
         "required": False, "description": "Small constant for numerical stability"},
    ],
    "type_constraints": [
        {"type_var": "T",
         "allowed": ["tensor(float16)", "tensor(float)", "tensor(double)"],
         "description": "Constrain input and output types to float tensors"},
    ],
    "shape_rule": "output.shape == input[0].shape",
    "doc": "Computes Root Mean Square Layer Normalization: Y = X / sqrt(mean(X^2, axis=-1) + eps) * gamma",
}

print("Custom Operator Schema:")
print(f"  Identity:  {custom_schema['domain']}::{custom_schema['op_type']} v{custom_schema['version']}")
print(f"  Inputs:    {[(i['name'], i['type_var'], i['arity']) for i in custom_schema['inputs']]}")
print(f"  Outputs:   {[(o['name'], o['type_var'], o['arity']) for o in custom_schema['outputs']]}")
print(f"  Attrs:     {[(a['name'], a['type'], a['default']) for a in custom_schema['attributes']]}")
print(f"  Types:     {custom_schema['type_constraints'][0]['allowed']}")
print(f"  Shape:     {custom_schema['shape_rule']}")
print(f"  Doc:       {custom_schema['doc']}")

<a id='3-registration-model-side'></a>
## 3. Registration: Model-Side Declaration

On the **model side**, a custom op is declared by:

1. Adding an `opset_import` entry for the custom domain and version
2. Creating `NodeProto` instances with the custom domain and op_type
3. Setting any custom attributes on the node

### Model-Side Architecture

```
  ┌──────────────────────────────────────────────────────────┐
  │  ModelProto                                               │
  │                                                           │
  │  opset_import:                                            │
  │    ["": 17]                 ← standard ops                │
  │    ["com.tutorial.ops": 1]  ← our custom domain          │
  │                                                           │
  │  ┌────────────────────────────────────────────────────┐  │
  │  │ GraphProto                                         │  │
  │  │                                                    │  │
  │  │  Node(op="Conv", domain="")                        │  │
  │  │    │                                               │  │
  │  │    ▼                                               │  │
  │  │  Node(op="RmsNorm", domain="com.tutorial.ops")     │  │
  │  │    │  attrs: {epsilon: 1e-6}                       │  │
  │  │    ▼                                               │  │
  │  │  Node(op="MatMul", domain="")                      │  │
  │  │                                                    │  │
  │  └────────────────────────────────────────────────────┘  │
  └──────────────────────────────────────────────────────────┘
```

The model file is **self-contained** in terms of declaring custom ops — it names the
domain and op_type. However, it does **not** contain the kernel implementation. That
must be registered separately in the runtime.

In [ ]:
# Create a model with a custom domain operator
CUSTOM_DOMAIN = "com.tutorial.ops"
CUSTOM_OPSET = 1

# Custom RmsNorm node
custom_node = helper.make_node(
    "RmsNorm",
    inputs=["X", "gamma"],
    outputs=["Y"],
    domain=CUSTOM_DOMAIN,
    epsilon=1e-6,
)

# Graph I/O
hidden = 64
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", "seq", hidden])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", "seq", hidden])
gamma = numpy_helper.from_array(
    np.ones(hidden, dtype=np.float32), "gamma")

graph = helper.make_graph([custom_node], "custom_rmsnorm", [X], [Y],
                          initializer=[gamma])
model = helper.make_model(
    graph,
    opset_imports=[
        helper.make_opsetid("", 17),
        helper.make_opsetid(CUSTOM_DOMAIN, CUSTOM_OPSET),
    ])

print("Model with custom operator:")
print(f"  OpSet imports:")
for oi in model.opset_import:
    d = oi.domain if oi.domain else '"" (default)'
    print(f"    domain={d}, version={oi.version}")

print(f"\n  Custom node details:")
node = model.graph.node[0]
print(f"    op_type:  {node.op_type}")
print(f"    domain:   {node.domain}")
print(f"    inputs:   {list(node.input)}")
print(f"    outputs:  {list(node.output)}")
print(f"    attrs:    {[(a.name, a.f) for a in node.attribute]}")

# Save the model
model_path = "/tmp/custom_rmsnorm.onnx"
onnx.save(model, model_path)
print(f"\n  Saved to: {model_path}")
print(f"  Size: {len(model.SerializeToString()):,} bytes")

<a id='4-registration-runtime-side'></a>
## 4. Registration: Runtime-Side Kernels

On the **runtime side**, a custom op requires a kernel implementation that the runtime
can invoke. The registration creates a mapping:

$$(\texttt{domain}, \texttt{op\_type}, \texttt{version}) \;\mapsto\; \text{KernelFunction}$$

### Two-Layer Registration Architecture

```
  ONNX Model File                    ONNX Runtime
  ┌──────────────┐                  ┌──────────────────────────────┐
  │              │                  │  Kernel Registry              │
  │  NodeProto:  │                  │                               │
  │  domain=     │    load model    │  ┌─────────────────────────┐ │
  │  "com.x.ops"│ ──────────────▶  │  │ (com.x.ops, RmsNorm, 1)│ │
  │  op_type=   │                  │  │    → RmsNormKernel()    │ │
  │  "RmsNorm"  │                  │  └─────────────────────────┘ │
  │              │                  │                               │
  │  (declares   │                  │  (provides                    │
  │   the what)  │                  │   the how)                    │
  └──────────────┘                  └──────────────────────────────┘
```

### ONNX Runtime Custom Op Registration (C++ API)

In ONNX Runtime, custom ops are registered through the C/C++ API:

1. **Define the kernel class** — implements `Compute()` method
2. **Define the custom op struct** — specifies name, domain, inputs/outputs
3. **Create a custom op domain** — groups related custom ops
4. **Register with session options** — tells ORT to load the custom ops

### Python-Side Simulation

For development and testing, we can simulate custom op execution using Python.
The `ReferenceEvaluator` supports custom op implementations via Python classes.

In [ ]:
# Simulate a custom op kernel in Python for ReferenceEvaluator
from onnx.reference import ReferenceEvaluator
from onnx.reference.op_run import OpRun

class RmsNorm(OpRun):
    """Python reference implementation of RmsNorm custom op."""
    op_domain = "com.tutorial.ops"

    def _run(self, X, gamma, epsilon=1e-6):
        rms = np.sqrt(np.mean(X ** 2, axis=-1, keepdims=True) + epsilon)
        return (X / rms * gamma,)

# Build model with custom op
X_info = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 4, 64])
Y_info = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [2, 4, 64])
gamma_init = numpy_helper.from_array(np.ones(64, dtype=np.float32), "gamma")

custom_node = helper.make_node(
    "RmsNorm", ["X", "gamma"], ["Y"],
    domain="com.tutorial.ops", epsilon=1e-6)

graph = helper.make_graph([custom_node], "test_custom", [X_info], [Y_info],
                          initializer=[gamma_init])
model = helper.make_model(
    graph, opset_imports=[
        helper.make_opsetid("", 17),
        helper.make_opsetid("com.tutorial.ops", 1)])

try:
    ev = ReferenceEvaluator(model, new_ops=[RmsNorm])
    x = np.random.randn(2, 4, 64).astype(np.float32)
    y = ev.run(None, {"X": x})[0]
    
    rms_expected = x / np.sqrt(np.mean(x**2, axis=-1, keepdims=True) + 1e-6)
    print(f"Custom op execution successful!")
    print(f"  Input shape:  {x.shape}")
    print(f"  Output shape: {y.shape}")
    print(f"  Matches expected: {np.allclose(y, rms_expected, atol=1e-5)}")
    print(f"  Output mean: {y.mean():.6f}")
    print(f"  Output std:  {y.std():.6f}")
except Exception as e:
    print(f"Custom op execution: {e}")

<a id='5-type-inference-rules'></a>
## 5. Type Inference Rules

Type inference determines the **element type** of each tensor in the graph.
For custom ops, you must define how types propagate from inputs to outputs.

### Formal Type Propagation

Given type constraint $T \in \{\text{float16}, \text{float32}, \text{float64}\}$
and inputs bound to type variable $T$:

$$\text{type}(\text{output}) = \text{bind}(T, \text{type}(\text{input\_0}))$$

### Type Binding Rules

```
  Input X: float32  ──┐
                       ├──▶ T binds to float32
  Input gamma: float32 ──┘       │
                                 │
  Type constraint check:         │
    float32 ∈ {float16, float32, float64}?  ✓
                                 │
  Output Y: T = float32  ◀───────┘
```

### Multiple Type Variables

Some operators use multiple type variables for heterogeneous typing:

$$T_1 \in \{\text{float32}, \text{float64}\} \quad \text{(data types)}$$
$$T_2 \in \{\text{int32}, \text{int64}\} \quad \text{(index types)}$$

### Cross-Input Type Consistency

When multiple inputs share the same type variable $T$, they **must** have the same
element type:

$$\text{type}(X) = T \;\wedge\; \text{type}(\gamma) = T \;\Rightarrow\; \text{type}(X) = \text{type}(\gamma)$$

In [ ]:
# Demonstrate type inference and validation
def check_type_consistency(model):
    """Check type consistency across graph tensors."""
    type_map = {}
    
    for inp in model.graph.input:
        elem_type = inp.type.tensor_type.elem_type
        type_map[inp.name] = TensorProto.DataType.Name(elem_type)
    
    for init in model.graph.initializer:
        type_map[init.name] = TensorProto.DataType.Name(init.data_type)
    
    for out in model.graph.output:
        elem_type = out.type.tensor_type.elem_type
        if elem_type:
            type_map[out.name] = TensorProto.DataType.Name(elem_type)
    
    return type_map

# Build models with different types to test type propagation
for dtype, dtype_name in [(TensorProto.FLOAT, "float32"),
                           (TensorProto.DOUBLE, "float64"),
                           (TensorProto.FLOAT16, "float16")]:
    X = helper.make_tensor_value_info("X", dtype, [2, 4, 8])
    Y = helper.make_tensor_value_info("Y", dtype, [2, 4, 8])
    gamma = numpy_helper.from_array(
        np.ones(8, dtype=np.float32 if dtype != TensorProto.DOUBLE else np.float64),
        "gamma")
    
    node = helper.make_node("Relu", ["X"], ["Y"])
    graph = helper.make_graph([node], f"type_test_{dtype_name}", [X], [Y])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    
    type_map = check_type_consistency(model)
    print(f"{dtype_name}: {type_map}")

# Show type constraints for standard ops
for op_name in ["Relu", "MatMul", "Conv", "Cast"]:
    schema = defs.get_schema(op_name, defs.onnx_opset_version())
    print(f"\n{op_name} type constraints:")
    for tc in schema.type_constraints:
        types = list(tc.allowed_type_strs)[:5]
        more = f" (+{len(tc.allowed_type_strs)-5})" if len(tc.allowed_type_strs) > 5 else ""
        print(f"  {tc.type_param_str}: {types}{more}")

<a id='6-shape-inference-rules'></a>
## 6. Shape Inference Rules

Shape inference propagates tensor shapes through the graph **without executing it**.
For custom ops, shape inference is critical for downstream optimization.

### Shape Inference Categories

| Category | Rule | Example |
|----------|------|---------|
| **Identity** | $\text{shape}(Y) = \text{shape}(X)$ | Relu, Sigmoid, BN |
| **Broadcast** | $\text{shape}(Y) = \text{broadcast}(X_1, X_2)$ | Add, Mul |
| **Reduction** | $\text{shape}(Y) = \text{drop\_axis}(X, k)$ | ReduceSum |
| **Reshape** | $\text{shape}(Y) = \text{target\_shape}$ | Reshape, Flatten |
| **Conv** | $d_{\text{out}} = \lfloor (d_{\text{in}} + 2p - d(k-1) - 1)/s + 1 \rfloor$ | Conv |
| **MatMul** | $[\ldots, M, K] \times [\ldots, K, N] \to [\ldots, M, N]$ | MatMul |
| **Custom** | User-defined function | Custom ops |

### For Custom Ops

Since the ONNX framework doesn't know your custom op's semantics, shape inference
will **stop** at custom op nodes unless you provide explicit output shape information
in the model graph.

**Options for shape inference with custom ops:**
1. Explicitly set output shapes in `ValueInfoProto` for all custom op outputs
2. Register a shape inference function (C++ API in ORT)
3. Use `FunctionProto` which decomposes to standard ops (shapes inferred automatically)

In [ ]:
# Demonstrate shape inference behavior with and without custom ops

# Standard ops: shape inference works automatically
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 3, 32, 32])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)  # unknown shape
W = numpy_helper.from_array(
    np.random.randn(16, 3, 3, 3).astype(np.float32) * 0.1, "W")

nodes = [
    helper.make_node("Conv", ["X", "W"], ["conv_out"],
                     kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
    helper.make_node("Relu", ["conv_out"], ["Y"]),
]

graph = helper.make_graph(nodes, "shape_demo", [X], [Y], initializer=[W])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

# Before shape inference
out_dims_before = model.graph.output[0].type.tensor_type.shape
print(f"Standard ops before inference: output shape = {out_dims_before}")

# After shape inference
model = shape_inference.infer_shapes(model)
out_shape = [d.dim_param if d.dim_param else d.dim_value
             for d in model.graph.output[0].type.tensor_type.shape.dim]
print(f"Standard ops after inference:  output shape = {out_shape}")

# With custom op: shape inference may not propagate
print(f"\nWith custom op (no shape inference function):")
X2 = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 64])
Y2 = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

custom_node = helper.make_node("MyOp", ["X"], ["Y"], domain="com.test")
graph2 = helper.make_graph([custom_node], "custom_shape", [X2], [Y2])
model2 = helper.make_model(
    graph2, opset_imports=[
        helper.make_opsetid("", 17),
        helper.make_opsetid("com.test", 1)])

try:
    model2 = shape_inference.infer_shapes(model2)
    out_shape2 = model2.graph.output[0].type.tensor_type.shape
    has_shape = out_shape2 is not None and len(out_shape2.dim) > 0
    print(f"  Shape inferred: {has_shape}")
    if has_shape:
        dims = [d.dim_param if d.dim_param else d.dim_value for d in out_shape2.dim]
        print(f"  Output shape: {dims}")
    else:
        print(f"  Shape unknown — must provide explicit output shapes")
except Exception as e:
    print(f"  Shape inference: {e}")

<a id='7-functionproto-alternative'></a>
## 7. FunctionProto as an Alternative

`FunctionProto` lets you define a custom operator as a **subgraph of standard ops**.
This gives you the naming convenience of a custom op while maintaining full portability.

### FunctionProto Architecture

```
  ┌─────────────────────────────────────────────────────────┐
  │  FunctionProto: "RmsNorm"                               │
  │  domain: "com.tutorial.ops"                              │
  │                                                          │
  │  Inputs: [X, gamma]                                      │
  │  Outputs: [Y]                                            │
  │  Attributes: [epsilon]                                   │
  │                                                          │
  │  Body (standard ops):                                    │
  │    ┌──────────┐                                          │
  │    │ Mul(X,X)  │──▶ x_sq                                 │
  │    └──────────┘                                          │
  │         │                                                │
  │    ┌──────────────┐                                      │
  │    │ ReduceMean   │──▶ mean_sq                           │
  │    └──────────────┘                                      │
  │         │                                                │
  │    ┌──────────┐                                          │
  │    │ Add(+eps) │──▶ stable                               │
  │    └──────────┘                                          │
  │         │                                                │
  │    ┌──────────┐                                          │
  │    │   Sqrt    │──▶ rms                                  │
  │    └──────────┘                                          │
  │         │                                                │
  │    ┌──────────┐                                          │
  │    │ Div(X/rms)│──▶ normed                               │
  │    └──────────┘                                          │
  │         │                                                │
  │    ┌──────────────┐                                      │
  │    │Mul(normed,γ) │──▶ Y                                 │
  │    └──────────────┘                                      │
  │                                                          │
  │  ★ Portable: any runtime can expand this to standard ops │
  │  ★ But: runtime can also fuse into optimized kernel      │
  └─────────────────────────────────────────────────────────┘
```

### Advantages of FunctionProto

| Property | Custom Op | FunctionProto |
|----------|-----------|---------------|
| Portability | Requires kernel on each runtime | Works everywhere |
| Performance | Can use optimized kernel | Runtime may fuse |
| Shape inference | Must provide | Automatic (from body) |
| Type inference | Must provide | Automatic (from body) |
| Debugging | Black box | Inspectable subgraph |

In [ ]:
# Build a FunctionProto for RmsNorm
def make_rmsnorm_function():
    """Create a FunctionProto that decomposes RmsNorm into standard ops."""
    func = helper.make_function(
        domain="com.tutorial.ops",
        fname="RmsNorm",
        inputs=["X", "gamma"],
        outputs=["Y"],
        nodes=[
            helper.make_node("Mul", ["X", "X"], ["x_sq"]),
            helper.make_node("ReduceMean", ["x_sq"], ["mean_sq"],
                             axes=[-1], keepdims=1),
            helper.make_node("Constant", [], ["eps_val"],
                             value=numpy_helper.from_array(
                                 np.array(1e-6, dtype=np.float32))),
            helper.make_node("Add", ["mean_sq", "eps_val"], ["stable"]),
            helper.make_node("Sqrt", ["stable"], ["rms"]),
            helper.make_node("Div", ["X", "rms"], ["normed"]),
            helper.make_node("Mul", ["normed", "gamma"], ["Y"]),
        ],
        opset_imports=[
            helper.make_opsetid("", 11),
        ],
    )
    return func

rmsnorm_func = make_rmsnorm_function()

print(f"FunctionProto: {rmsnorm_func.domain}::{rmsnorm_func.name}")
print(f"  Inputs:  {list(rmsnorm_func.input)}")
print(f"  Outputs: {list(rmsnorm_func.output)}")
print(f"  Body nodes: {len(rmsnorm_func.node)}")
for i, node in enumerate(rmsnorm_func.node):
    print(f"    [{i}] {node.op_type}: {list(node.input)} → {list(node.output)}")

# Build model using the function
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 4, 64])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
gamma_init = numpy_helper.from_array(np.ones(64, dtype=np.float32), "gamma")

call_node = helper.make_node("RmsNorm", ["X", "gamma"], ["Y"],
                              domain="com.tutorial.ops")
graph = helper.make_graph([call_node], "func_model", [X], [Y],
                          initializer=[gamma_init])
model = helper.make_model(
    graph,
    opset_imports=[
        helper.make_opsetid("", 17),
        helper.make_opsetid("com.tutorial.ops", 1)],
    functions=[rmsnorm_func])

print(f"\nModel with FunctionProto:")
print(f"  Functions: {len(model.functions)}")
print(f"  Graph nodes: {len(model.graph.node)}")
print(f"  Size: {len(model.SerializeToString()):,} bytes")

<a id='8-end-to-end-workflow'></a>
## 8. End-to-End Workflow

Creating and deploying a custom operator follows an **8-step lifecycle**:

```
  ┌─────────────────────────────────────────────────────────────┐
  │              CUSTOM OP LIFECYCLE (8 Steps)                   │
  ├─────────────────────────────────────────────────────────────┤
  │                                                             │
  │  Step 1: IDENTIFY                                           │
  │    └── Define the math, I/O tensor ranks, semantics         │
  │                                                             │
  │  Step 2: DESIGN SCHEMA                                      │
  │    └── Types, attributes, shape rules, constraints          │
  │                                                             │
  │  Step 3: PROTOTYPE                                          │
  │    └── Standard-op decomposition for parity tests           │
  │                                                             │
  │  Step 4: IMPLEMENT KERNEL                                   │
  │    └── CPU reference first, then GPU/EP-specific            │
  │                                                             │
  │  Step 5: REGISTER                                           │
  │    └── Map (domain, op_type) → kernel in runtime            │
  │                                                             │
  │  Step 6: EXPORT MODEL                                       │
  │    └── Reference custom domain & opset in model file        │
  │                                                             │
  │  Step 7: PACKAGE                                            │
  │    └── Ship runtime extension + schema docs + model         │
  │                                                             │
  │  Step 8: TEST & VALIDATE                                    │
  │    └── Golden vectors, fuzz shapes, cross-platform CI       │
  │                                                             │
  └─────────────────────────────────────────────────────────────┘
```

In [ ]:
# Complete end-to-end example: Swish activation (x * sigmoid(x))
# This was not a standard ONNX op until recently

DOMAIN = "com.tutorial.activations"
VERSION = 1

# Step 1 & 2: Identify and design schema
swish_schema = {
    "domain": DOMAIN,
    "op_type": "Swish",
    "doc": "Swish(x) = x * sigmoid(beta * x)",
    "inputs": [("X", "T")],
    "outputs": [("Y", "T")],
    "attributes": [("beta", "float", 1.0)],
    "type_constraints": {"T": ["tensor(float)", "tensor(double)"]},
}
print(f"Step 1-2: Schema defined for {DOMAIN}::Swish")

# Step 3: Prototype with standard ops
X_info = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", "dim"])
Y_info = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

proto_nodes = [
    helper.make_node("Sigmoid", ["X"], ["sig_x"]),
    helper.make_node("Mul", ["X", "sig_x"], ["Y"]),
]
proto_graph = helper.make_graph(proto_nodes, "swish_proto", [X_info], [Y_info])
proto_model = helper.make_model(proto_graph, opset_imports=[helper.make_opsetid("", 17)])
proto_model = shape_inference.infer_shapes(proto_model)
checker.check_model(proto_model)
print(f"Step 3: Prototype model built with {len(proto_nodes)} standard ops")

# Step 4: Implement kernel (Python reference)
class Swish(OpRun):
    op_domain = DOMAIN
    def _run(self, X, beta=1.0):
        return (X * (1.0 / (1.0 + np.exp(-beta * X))),)
print(f"Step 4: Kernel implemented (Python reference)")

# Step 5 & 6: Register and export
custom_node = helper.make_node("Swish", ["X"], ["Y"], domain=DOMAIN, beta=1.0)
graph = helper.make_graph([custom_node], "swish_model", [X_info], [Y_info])
model = helper.make_model(
    graph, opset_imports=[
        helper.make_opsetid("", 17),
        helper.make_opsetid(DOMAIN, VERSION)])
print(f"Step 5-6: Model exported with custom domain")

# Step 7: Package (save model)
onnx.save(model, "/tmp/swish_custom.onnx")
print(f"Step 7: Model packaged ({len(model.SerializeToString()):,} bytes)")

# Step 8: Test
try:
    ev_proto = ReferenceEvaluator(proto_model)
    ev_custom = ReferenceEvaluator(model, new_ops=[Swish])
    
    x_test = np.random.randn(4, 8).astype(np.float32)
    y_proto = ev_proto.run(None, {"X": x_test})[0]
    y_custom = ev_custom.run(None, {"X": x_test})[0]
    
    print(f"Step 8: Testing")
    print(f"  Prototype vs Custom match: {np.allclose(y_proto, y_custom, atol=1e-6)}")
    print(f"  Max error: {np.max(np.abs(y_proto - y_custom)):.2e}")
    print(f"  All steps complete!")
except Exception as e:
    print(f"Step 8: {e}")

<a id='9-mixed-models'></a>
## 9. Mixed Standard + Custom Op Models

In practice, custom ops are mixed with standard ops in the same graph. The graph
is a DAG where some nodes use the default domain and others use custom domains.

### Mixed Graph Architecture

```
  Input X [batch, seq, 768]
      │
      ▼
  ┌──────────────────┐
  │ MatMul           │  domain="" (standard)       [batch, seq, 768]
  │ X @ W_qkv        │
  └──────────────────┘
      │
      ▼
  ┌──────────────────┐
  │ Split            │  domain="" (standard)       3 × [batch, seq, 256]
  │ → Q, K, V        │
  └──────────────────┘
      │
      ▼
  ┌──────────────────┐
  │ FlashAttention   │  domain="com.vendor"        [batch, seq, 256]
  │ (CUSTOM OP)      │  (fused, optimized kernel)
  └──────────────────┘
      │
      ▼
  ┌──────────────────┐
  │ Add (residual)   │  domain="" (standard)       [batch, seq, 768]
  └──────────────────┘
      │
      ▼
  ┌──────────────────┐
  │ RmsNorm          │  domain="com.vendor"        [batch, seq, 768]
  │ (CUSTOM OP)      │
  └──────────────────┘
      │
      ▼
  Output [batch, seq, 768]
```

In [ ]:
# Build a model mixing standard and custom operators
CUSTOM_DOMAIN = "com.tutorial.ops"
hidden = 32

# Inputs/outputs
X_info = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", hidden])
Y_info = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", hidden])

# Initializers
W1 = numpy_helper.from_array(
    np.random.randn(hidden, hidden).astype(np.float32) * 0.02, "W1")
b1 = numpy_helper.from_array(np.zeros(hidden, dtype=np.float32), "b1")
gamma = numpy_helper.from_array(np.ones(hidden, dtype=np.float32), "gamma")
W2 = numpy_helper.from_array(
    np.random.randn(hidden, hidden).astype(np.float32) * 0.02, "W2")
b2 = numpy_helper.from_array(np.zeros(hidden, dtype=np.float32), "b2")

# Mixed node list: standard + custom + standard
nodes = [
    # Standard: linear layer 1
    helper.make_node("MatMul", ["X", "W1"], ["mm1"]),
    helper.make_node("Add", ["mm1", "b1"], ["linear1"]),

    # Custom: Swish activation
    helper.make_node("Swish", ["linear1"], ["act1"],
                     domain=CUSTOM_DOMAIN, beta=1.0),

    # Standard: residual connection
    helper.make_node("Add", ["X", "act1"], ["residual"]),

    # Custom: RmsNorm
    helper.make_node("RmsNorm", ["residual", "gamma"], ["normed"],
                     domain=CUSTOM_DOMAIN, epsilon=1e-6),

    # Standard: linear layer 2
    helper.make_node("MatMul", ["normed", "W2"], ["mm2"]),
    helper.make_node("Add", ["mm2", "b2"], ["Y"]),
]

graph = helper.make_graph(
    nodes, "mixed_model", [X_info], [Y_info],
    initializer=[W1, b1, gamma, W2, b2])
model = helper.make_model(
    graph, opset_imports=[
        helper.make_opsetid("", 17),
        helper.make_opsetid(CUSTOM_DOMAIN, 1)])

print(f"Mixed model:")
print(f"  Total nodes: {len(model.graph.node)}")
print(f"  Standard ops: {sum(1 for n in model.graph.node if not n.domain)}")
print(f"  Custom ops:   {sum(1 for n in model.graph.node if n.domain)}")
print(f"  Size: {len(model.SerializeToString()):,} bytes")

print(f"\n  Node details:")
for i, node in enumerate(model.graph.node):
    d = node.domain if node.domain else '""'
    print(f"    [{i}] {node.op_type:<12} domain={d:<25} "
          f"{list(node.input)} → {list(node.output)}")

# Execute with registered custom ops
try:
    ev = ReferenceEvaluator(model, new_ops=[Swish, RmsNorm])
    x = np.random.randn(4, hidden).astype(np.float32)
    y = ev.run(None, {"X": x})[0]
    print(f"\n  Execution successful!")
    print(f"  Input:  {x.shape} → Output: {y.shape}")
    print(f"  Output mean: {y.mean():.4f}, std: {y.std():.4f}")
except Exception as e:
    print(f"\n  Execution: {e}")

<a id='10-testing-patterns'></a>
## 10. Testing Patterns

Custom operators require rigorous testing since they lack the community-validated
test suites that standard operators enjoy.

### Testing Strategy

| Level | Test Type | What It Catches |
|-------|-----------|------------------|
| 1 | **Golden vectors** | Numerical correctness on known inputs/outputs |
| 2 | **Parity checks** | Matches standard-op decomposition or framework reference |
| 3 | **Shape fuzzing** | Edge cases: empty tensors, batch=1, rank=1 |
| 4 | **Type coverage** | Works with float16, float32, float64 |
| 5 | **Gradient check** | Numerical gradient vs analytical (if differentiable) |
| 6 | **Cross-platform** | Same results on CPU, GPU, different hardware |
| 7 | **Stress test** | Large tensors, many iterations, memory leaks |

### Golden Vector Testing

Pre-compute expected results from a trusted reference and store them:

$$\|y_{\text{custom}} - y_{\text{golden}}\|_\infty < \epsilon_{\text{tolerance}}$$

Typical tolerances:
- float32: $\epsilon = 10^{-5}$ to $10^{-6}$
- float16: $\epsilon = 10^{-2}$ to $10^{-3}$
- float64: $\epsilon = 10^{-12}$ to $10^{-14}$

In [ ]:
# Comprehensive testing pattern for custom operators
def test_custom_op(custom_impl, reference_impl, test_shapes, atol=1e-5):
    """Test a custom op against a reference implementation."""
    results = []
    
    for shape in test_shapes:
        x = np.random.randn(*shape).astype(np.float32)
        gamma = np.ones(shape[-1], dtype=np.float32)
        
        try:
            y_custom = custom_impl(x, gamma)
            y_ref = reference_impl(x, gamma)
            
            max_err = np.max(np.abs(y_custom - y_ref))
            passed = max_err < atol
            results.append({
                'shape': shape,
                'max_error': max_err,
                'passed': passed,
                'output_shape': y_custom.shape,
            })
        except Exception as e:
            results.append({
                'shape': shape,
                'error': str(e),
                'passed': False,
            })
    
    return results

# Reference and custom implementations
def rmsnorm_reference(x, gamma, eps=1e-6):
    rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + eps)
    return x / rms * gamma

def rmsnorm_custom(x, gamma, eps=1e-6):
    rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + eps)
    return x / rms * gamma

# Test 1: Golden vectors
test_shapes = [
    (1, 1, 8),       # minimal
    (2, 4, 64),      # typical
    (16, 128, 768),  # large (transformer-scale)
    (1, 1, 1),       # edge: single element
    (100, 1, 32),    # edge: seq_len=1
]

print("Test 1: Golden Vector Tests")
print(f"{'Shape':<20} │ {'Max Error':>12} │ {'Status':>8}")
print("─" * 48)
results = test_custom_op(rmsnorm_custom, rmsnorm_reference, test_shapes)
for r in results:
    if 'error' in r:
        print(f"{str(r['shape']):<20} │ {'ERROR':>12} │ {'FAIL':>8}")
    else:
        status = 'PASS' if r['passed'] else 'FAIL'
        print(f"{str(r['shape']):<20} │ {r['max_error']:>12.2e} │ {status:>8}")

# Test 2: Fuzz testing with random shapes
print(f"\nTest 2: Shape Fuzzing (20 random shapes)")
np.random.seed(42)
fuzz_shapes = []
for _ in range(20):
    rank = np.random.randint(2, 5)
    shape = tuple(np.random.randint(1, 32, size=rank))
    fuzz_shapes.append(shape)

fuzz_results = test_custom_op(rmsnorm_custom, rmsnorm_reference, fuzz_shapes)
passed = sum(1 for r in fuzz_results if r['passed'])
total = len(fuzz_results)
print(f"  Passed: {passed}/{total}")
failures = [r for r in fuzz_results if not r['passed']]
if failures:
    print(f"  Failed shapes: {[r['shape'] for r in failures]}")

<a id='11-security-considerations'></a>
## 11. Security Considerations

Custom operators introduce **significant security surface** because they involve
loading and executing arbitrary code.

### Threat Model

```
  ┌──────────────────────────────────────────────────────────────┐
  │  SECURITY RISKS WITH CUSTOM OPS                              │
  ├──────────────────────────────────────────────────────────────┤
  │                                                              │
  │  1. CODE EXECUTION                                           │
  │     Custom op .so/.dll loading = arbitrary code execution    │
  │     Risk: malicious model + matching .so = full compromise   │
  │                                                              │
  │  2. SUPPLY CHAIN                                             │
  │     Model files from untrusted sources may reference         │
  │     custom domains that trigger loading malicious libraries   │
  │                                                              │
  │  3. DENIAL OF SERVICE                                        │
  │     Custom ops with unbounded memory allocation              │
  │     or infinite loops                                        │
  │                                                              │
  │  4. DATA EXFILTRATION                                        │
  │     Custom op kernel could transmit input data externally    │
  │                                                              │
  └──────────────────────────────────────────────────────────────┘
```

### Mitigations

| Risk | Mitigation |
|------|------------|
| Arbitrary code execution | Whitelist allowed custom op libraries; sign .so files |
| Supply chain attacks | Validate model provenance; scan for unknown domains |
| Resource exhaustion | Sandbox custom op execution; set memory/time limits |
| Data exfiltration | Network isolation; audit custom op source code |
| Version confusion | Pin exact domain versions; reject unknown versions |

In [ ]:
# Security audit: scan a model for custom ops
def audit_model_domains(model):
    """Audit a model for custom domain usage."""
    findings = []
    
    # Check opset imports
    known_domains = {'', 'ai.onnx', 'ai.onnx.ml', 'ai.onnx.training',
                     'ai.onnx.preview.training'}
    
    for oi in model.opset_import:
        if oi.domain not in known_domains:
            findings.append({
                'type': 'CUSTOM_DOMAIN',
                'severity': 'WARNING',
                'detail': f"Custom domain imported: {oi.domain!r} v{oi.version}",
            })
    
    # Check nodes
    custom_nodes = []
    for node in model.graph.node:
        d = node.domain if node.domain else ''
        if d not in known_domains:
            custom_nodes.append(node)
            findings.append({
                'type': 'CUSTOM_OP',
                'severity': 'WARNING',
                'detail': f"Custom op: {d}::{node.op_type}",
            })
    
    return findings

# Audit our mixed model
print("Security Audit Report")
print("=" * 50)
findings = audit_model_domains(model)

if not findings:
    print("  No custom domains found — model uses only standard ops.")
else:
    for f in findings:
        print(f"  [{f['severity']}] {f['type']}: {f['detail']}")
    print(f"\n  Total findings: {len(findings)}")
    print(f"  Action: Verify custom op sources before deployment")

<a id='12-key-takeaways'></a>
## 12. Key Takeaways

1. **Custom ops trade portability for expressiveness/performance.** Use them only when
   standard ops or FunctionProto decompositions are insufficient.

2. A custom op requires **two registrations**: model-side (domain + op_type in the graph)
   and runtime-side (kernel implementation in the execution engine).

3. The schema contract defines:
   $$S(O) = (\mathcal{I}, \mathcal{O}, \mathcal{A}, \mathcal{T}, \mathcal{F}_{\text{shape}}, \mathcal{D})$$
   Inputs, outputs, attributes, type constraints, shape inference, and documentation.

4. **FunctionProto** is the preferred alternative — it decomposes a named op into
   standard ops, giving you custom naming with full portability.

5. **Type inference** propagates through type variables: inputs sharing a type variable
   $T$ must have the same element type.

6. **Shape inference** stops at custom ops unless you provide explicit output shapes
   or register a shape inference function.

7. **Testing** is critical: use golden vectors, parity checks against standard-op
   decompositions, shape fuzzing, and cross-platform validation.

8. **Security**: custom op .so loading is code execution. Whitelist allowed libraries,
   validate model provenance, and audit custom op source code.

9. **Version your custom domain** independently from the standard ONNX opset.
   Never mutate schema semantics in place — always bump the version.

10. Always provide a **standard-op decomposition** alongside your custom op for
    parity testing and fallback on unsupported runtimes.